# gCRL-AE — Simple Example (Simulated Data)

Trains gCRL-AE on a small synthetic dataset to verify the pipeline end-to-end.

**Pipeline overview**

| Step | Description |
|------|-------------|
| 0 | Load example AnnData; compute eigengenes; inspect explained variance |
| 1 | Train gCRL-AE; inspect loss curve |
| 2 | Run partial-MCC permutation experiment; report significance |

**Input files** (under `data/example/`)
- `adata.h5ad` — synthetic single-cell AnnData with TF communities pre-assigned

**Output files** (under `results/simulated/example/`)
- `ae/` — model weights and embeddings (written by `train_gcrl_ae`)
- `partial_mcc/partial_mcc_results.npz` — real and permuted MCC scores

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import ttest_1samp

from gcrl.alignment.partial_mcc_perm_experiments import run_partial_mcc_perm_experiments
from gcrl.grn.eigengenes import compute_eigengenes
from gcrl.training.train_gcrl_ae import train_gcrl_ae

In [ ]:
# ── Control panel ────────────────────────────────────────────────

# --- paths ---
input_data_file = '../../data/example/adata.h5ad'
output_ae_dir   = '../../results/simulated/example/ae'
output_mcc_dir  = '../../results/simulated/example/partial_mcc'

# --- eigengene computation ---
eigengene_mode  = 'all_cells'   # 'all_cells' | 'global' (reference-based)
reference_query = 'intervention == "unperturbed"'

# --- AE architecture ---
ae_input_mode      = 'TF'    # 'TF' | 'ALL'
ae_reconstruct_all = True
ae_latent_dim      = None    # None → inferred as (#communities + 1)
ae_hidden_dims     = (64,)

# --- AE training ---
ae_standardize  = 'zscore'   # 'zscore' | 'zscore_ref' | 'minmax_0_1' | 'none'
ae_batch_size   = 256
ae_lr           = 1e-3
ae_num_epochs   = 100
ae_lr_step      = 100
ae_weight_decay = 1e-3
ae_val_frac     = 0.1

# --- partial-MCC permutation experiment ---
mcc_n_real_seeds   = 20
mcc_n_permutations = 20
mcc_n_perm_seeds   = 3
mcc_lr             = 5e-2
mcc_steps          = 100
mcc_master_seed    = 123
mcc_include_pooled = True

## 0. Load data and compute eigengenes

In [ ]:
adata = sc.read_h5ad(input_data_file)
print(adata)

In [ ]:
compute_eigengenes(
    adata,
    mode=eigengene_mode,
    reference_query=reference_query,
)
print(adata)

In [ ]:
meta     = adata.uns['comm_eig_meta']
comm_ids = adata.uns['X_comm_eig_comm_ids']

flat_stats  = meta['per_comm_stats']
flat_genes  = meta['per_comm_genes']
flat_pooled = meta['pooled_tf_stats']['explained_variance_ratio']

rows = []
for comm in comm_ids:
    rows.append({
        'community': comm,
        'n_tfs': len(flat_genes[str(comm)]),
        'explained_variance_ratio': flat_stats[str(comm)]['explained_variance_ratio'],
    })
rows.append({'community': 'all_TF',
             'n_tfs': sum(len(flat_genes[str(c)]) for c in comm_ids),
             'explained_variance_ratio': flat_pooled})

df_evr = pd.DataFrame(rows).set_index('community')
df_evr['explained_variance_ratio'] = (
    df_evr['explained_variance_ratio'] * 100
).round(1).astype(str) + '%'
print(df_evr.to_string())

## 1. Train gCRL-AE

In [ ]:
res = train_gcrl_ae(
    adata=adata,
    input_mode=ae_input_mode,
    reconstruct_all=ae_reconstruct_all,
    latent_dim=ae_latent_dim,
    hidden_dims=ae_hidden_dims,
    standardize=ae_standardize,
    reference_query=reference_query,
    batch_size=ae_batch_size,
    lr=ae_lr,
    num_epochs=ae_num_epochs,
    lr_step=ae_lr_step,
    weight_decay=ae_weight_decay,
    val_frac=ae_val_frac,
    device=None,
    outdir=output_ae_dir,
)

In [ ]:
loss     = res.history['loss']
val_loss = res.history.get('val_loss')

header = f"{'Epoch':>6}  {'Train loss':>12}  {'Val loss':>12}"
print(header)
print('-' * len(header))
for ep in [1, 10, 50, 100]:
    if ep > len(loss):
        break
    vl = f'{val_loss[ep-1]:12.4f}' if val_loss else '           -'
    print(f'{ep:6d}  {loss[ep-1]:12.4f}  {vl}')

print()
print(f'Train improvement: {loss[0]-loss[-1]:.4f}  ({100*(loss[0]-loss[-1])/loss[0]:.1f}% reduction)')
if val_loss:
    gap = val_loss[-1] - loss[-1]
    print(f'Final val - train gap: {gap:+.4f}  '
          f'({"possible overfitting" if gap > 0.25 else "no significant overfitting"})')

B = res.embeddings
print(f'\nEmbeddings shape: {B.shape}')

## 2. Partial-MCC permutation experiment

In [ ]:
os.makedirs(output_mcc_dir, exist_ok=True)

out = run_partial_mcc_perm_experiments(
    adata=adata,
    embeddings=B,
    community_col='community',
    reference_query=reference_query,
    mode=eigengene_mode,
    n_real_seeds=mcc_n_real_seeds,
    n_permutations=mcc_n_permutations,
    n_perm_seeds=mcc_n_perm_seeds,
    include_pooled_tf=mcc_include_pooled,
    lr=mcc_lr,
    steps=mcc_steps,
    device=None,
    master_seed=mcc_master_seed,
    save_density_path=os.path.join(output_mcc_dir, 'dens.png'),
)

np.savez(
    os.path.join(output_mcc_dir, 'partial_mcc_results.npz'),
    scores_real=out['scores_real'],
    scores_perm=out['scores_perm'],
)
print(f'Saved → {output_mcc_dir}/partial_mcc_results.npz')

In [ ]:
scores_real = out['scores_real']
scores_perm = out['scores_perm']

mu_real    = scores_real.mean()
perm_means = scores_perm.mean(axis=1)

print(f'Real   — mean: {mu_real:.4f}  std: {scores_real.std():.4f}  n={len(scores_real)}')
print(f'Perm   — mean: {perm_means.mean():.4f}  std: {perm_means.std():.4f}  n={len(perm_means)}')

t_stat, p_val = ttest_1samp(perm_means, popmean=mu_real, alternative='less')
print(f'\nOne-sample t-test (perm means < real mean): t = {t_stat:.4f},  p = {p_val:.4g}')